# Fabric 02 — Bronze Layer
**Purpose:** ingest raw Parquet into a Delta table in the Lakehouse. Add lineage metadata.
**Source:** Files/yellow_tripdata_2024-01.parquet
**Output table:** Tables/bronze_yellow_taxi (managed Delta in attached Lakehouse)

In [5]:
import uuid
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

LOG_TABLE = "pipeline_log"   # <-- Fabric: lakehouse-scoped, no schema prefix
RUN_ID = str(uuid.uuid4())
PIPELINE_NAME = "nyc_taxi_yellow"

LOG_SCHEMA = StructType([
    StructField("pipeline_name", StringType(),    nullable=False),
    StructField("run_id",        StringType(),    nullable=False),
    StructField("stage",         StringType(),    nullable=False),
    StructField("rows_in",       LongType(),      nullable=True),
    StructField("rows_out",      LongType(),      nullable=True),
    StructField("status",        StringType(),    nullable=False),
    StructField("error_message", StringType(),    nullable=True),
    StructField("run_timestamp", TimestampType(), nullable=False),
])

def log_pipeline_run(stage, rows_in, rows_out, status, error_message=None):
    log_row = spark.createDataFrame(
        [(PIPELINE_NAME, RUN_ID, stage, rows_in, rows_out, status, error_message, datetime.utcnow())],
        schema=LOG_SCHEMA
    )
    (log_row.write.format("delta").mode("append").saveAsTable(LOG_TABLE))
    print(f"[{stage}] {status} | rows_in={rows_in:,} rows_out={rows_out:,}")

StatementMeta(, 7ce08cd0-647d-47dd-9c97-e276013edb25, 7, Finished, Available, Finished, False)

In [6]:
from pyspark.sql import functions as F

SOURCE_PATH  = "Files/yellow_tripdata_2024-01 (2).parquet"
BRONZE_TABLE = "bronze_yellow_taxi"

try:
    df_raw = spark.read.parquet(SOURCE_PATH)
    rows_in = df_raw.count()

    df_bronze = (
        df_raw
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.lit(SOURCE_PATH))
    )

    (df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_TABLE))

    rows_out = spark.table(BRONZE_TABLE).count()
    log_pipeline_run("bronze", rows_in, rows_out, "SUCCESS")

except Exception as e:
    log_pipeline_run("bronze", -1, 0, "FAILED", str(e))
    raise

StatementMeta(, 7ce08cd0-647d-47dd-9c97-e276013edb25, 8, Finished, Available, Finished, False)

[bronze] SUCCESS | rows_in=2,964,624 rows_out=2,964,624


In [3]:
spark.sql(f"SELECT ingestion_timestamp, source_file, VendorID, tpep_pickup_datetime, fare_amount FROM {BRONZE_TABLE} LIMIT 5").show(truncate=False)
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").show(truncate=False)

StatementMeta(, 7ce08cd0-647d-47dd-9c97-e276013edb25, 5, Finished, Available, Finished, False)

+--------------------------+-----------------------------------------+--------+--------------------+-----------+
|ingestion_timestamp       |source_file                              |VendorID|tpep_pickup_datetime|fare_amount|
+--------------------------+-----------------------------------------+--------+--------------------+-----------+
|2026-05-05 10:01:00.040285|Files/yellow_tripdata_2024-01 (2).parquet|2       |2024-01-01 00:57:55 |17.7       |
|2026-05-05 10:01:00.040285|Files/yellow_tripdata_2024-01 (2).parquet|1       |2024-01-01 00:03:00 |10.0       |
|2026-05-05 10:01:00.040285|Files/yellow_tripdata_2024-01 (2).parquet|1       |2024-01-01 00:17:06 |23.3       |
|2026-05-05 10:01:00.040285|Files/yellow_tripdata_2024-01 (2).parquet|1       |2024-01-01 00:36:38 |10.0       |
|2026-05-05 10:01:00.040285|Files/yellow_tripdata_2024-01 (2).parquet|1       |2024-01-01 00:46:51 |7.9        |
+--------------------------+-----------------------------------------+--------+-----------------

In [7]:
spark.sql("""
    SELECT pipeline_name, run_id, stage, rows_in, rows_out, status, error_message, run_timestamp
    FROM pipeline_log
    ORDER BY run_timestamp DESC
""").show(truncate=False)

StatementMeta(, 7ce08cd0-647d-47dd-9c97-e276013edb25, 9, Finished, Available, Finished, False)

+---------------+------------------------------------+------+-------+--------+-------+-------------+--------------------------+
|pipeline_name  |run_id                              |stage |rows_in|rows_out|status |error_message|run_timestamp             |
+---------------+------------------------------------+------+-------+--------+-------+-------------+--------------------------+
|nyc_taxi_yellow|7c7e29fc-a5f7-46c0-92c7-37b9de7fb2a5|bronze|2964624|2964624 |SUCCESS|NULL         |2026-05-05 10:14:18.938109|
+---------------+------------------------------------+------+-------+--------+-------+-------------+--------------------------+

